In [20]:
#r "../Polars.NET.Core/bin/Debug/net8.0/Polars.NET.Core.dll"
#r "../Polars.CSharp/bin/Debug/net8.0/Polars.CSharp.dll"

using Microsoft.DotNet.Interactive.Formatting;
using Polars.CSharp;
using static Polars.CSharp.Polars;

Formatter.Register<DataFrame>((df,writer)=>
{
    writer.Write(df.ToHtml());
}, "text/html");

In [21]:
var df = DataFrame.FromColumns(new 
{
    Date = new[] { "2023-01-01", "2023-01-01","2023-01-02" }, // Index 0
    City = new[] { "London", "Manchester","London" },    // Index 1
    Temperature = new[] { 10.5, 9.0,12.1 },   // Index 2
    Rain = new[] {true,true,false} // Index 3
})
.WithColumns(Col("Date").Str.ToDate("%Y-%m-%d"));

df

Datedate,Citystr,Temperaturef64,Rainbool
2023/1/1,London,10.5,True
2023/1/1,Manchester,9,True
2023/1/2,London,12.1,False


In [22]:
var df_converted = df.Filter(Col("City") == "London")
                    .Select(Col("Date"),(Col("Temperature") * 1.8 + 32).Alias("Temp_F"));

df_converted

Datedate,Temp_Ff64
2023/1/1,50.9
2023/1/2,53.78


In [23]:
var LondonDf = df.Filter(Col("City") == "London");

LondonDf

Datedate,Citystr,Temperaturef64,Rainbool
2023/1/1,London,10.5,True
2023/1/2,London,12.1,False


In [24]:
var AggDf = df.
        GroupBy(Col("City"))
        .Agg(Col("Temperature").Mean().Alias("Avg_Temp")
            ,Col("Temperature").Max().Alias("Max_Temp")
            ,Col("Rain").Sum().Alias("Rainy_Days")
            ,Col("City").Count().Alias("Total_Records"));

AggDf

Citystr,Avg_Tempf64,Max_Tempf64,Rainy_Daysu32,Total_Recordsu32
London,11.3,12.1,1,2
Manchester,9,9,1,1


In [25]:
var WindowDf = df.Select(Col("Date"),Col("City"),Col("Temperature")
                        ,Col("Temperature").Mean().Over(Col("Date")).Alias("Daily_Avg")
                        ,(Col("Temperature") - Col("Temperature").Mean().Over(Col("Date"))).Alias("Diff")
        ).Sort("Date",false);

WindowDf

Datedate,Citystr,Temperaturef64,Daily_Avgf64,Difff64
2023/1/1,London,10.5,9.75,0.75
2023/1/1,Manchester,9,9.75,-0.75
2023/1/2,London,12.1,12.1,0


In [26]:
var lf = 
    df.Lazy()
        .Filter(Col("Temperature") > 10.0)
        .GroupBy("City")
        .Agg(Col("Temperature").Mean().Alias("Lazy_Avg_Temp"));

System.Console.WriteLine(lf.Explain(true));

var finalDf = lf.Collect();

finalDf

AGGREGATE[maintain_order: true]
  [col("Temperature").mean().alias("Lazy_Avg_Temp")] BY [col("City")]
  FROM
  FILTER [(col("Temperature")) > (10.0)]
  FROM
    DF ["Date", "City", "Temperature", "Rain"]; PROJECT["Temperature", "City"] 2/4 COLUMNS


Citystr,Lazy_Avg_Tempf64
London,11.3
